# Modern Coding Practice — Morse Code (Amazon FAR style)

**Format these interviews use:** one problem domain, then the interviewer keeps *adding constraints* on top of your working solution. Each part should build on the last without throwing the previous away. They watch how you:

1. Pin down ambiguity *before* coding (ask about inputs, error cases, scale).
2. Pick data structures that won't need to be ripped out at the next step.
3. Keep the code testable as it grows.
4. Reason about concurrency correctly (this part trips most people).

**How to use this notebook:** fill in the stubs, run the test cell under each part. Don't peek at the reference cells (collapsed at the bottom) until you've attempted it. Treat each part as a fresh 10–15 min round.

---

## Part 1 — Encode & Decode

Implement `encode(text)` and `decode(morse)`.

**Spec to lock down (say these out loud in the interview):**
- Letters separated by a single space, words separated by ` / ` (space-slash-space) — a common convention. Confirm with interviewer.
- Case-insensitive input; encode uppercases.
- Define behavior for unsupported characters: raise vs. skip. We'll raise `ValueError`.
- `decode(encode(x)) == x.upper()` for any supported `x` (round-trip property).


In [11]:
MORSE = {
    'A': '.-',    'B': '-...',  'C': '-.-.',  'D': '-..',   'E': '.',
    'F': '..-.',  'G': '--.',   'H': '....',  'I': '..',    'J': '.---',
    'K': '-.-',   'L': '.-..',  'M': '--',    'N': '-.',    'O': '---',
    'P': '.--.',  'Q': '--.-',  'R': '.-.',   'S': '...',   'T': '-',
    'U': '..-',   'V': '...-',  'W': '.--',   'X': '-..-',  'Y': '-.--',
    'Z': '--..',
    '0': '-----', '1': '.----', '2': '..---', '3': '...--', '4': '....-',
    '5': '.....', '6': '-....', '7': '--...', '8': '---..', '9': '----.',
}
# Build the reverse map once. Note: morse is a prefix-free-ish code only with separators,
# which is exactly why the space-separated format matters for decode.
INV_MORSE = {v: k for k, v in MORSE.items()}


def encode(text: str) -> str:
    res = []
    for c in text:
        c = c.upper()
        if c == ' ':
            res.append('/')
        elif c in MORSE:
            res.append(MORSE[c])
        else:
            raise ValueError(f"unsupported input character:{c}")
    return ' '.join(res)
    


def decode(morse: str) -> str:
    words = morse.split(' / ')
    text = []
    for word in words:
        chars = word.split(' ')
        for c in chars:
            text += INV_MORSE[c]
        text += ' '
    text.pop()
    print(''.join(text))
    return ''.join(text)

In [12]:
# --- Part 1 tests ---
def _t1():
    assert encode('SOS') == '... --- ...'
    assert encode('HELLO WORLD') == '.... . .-.. .-.. --- / .-- --- .-. .-.. -..'
    assert decode('... --- ...') == 'SOS'
    assert decode(encode('Hello World')) == 'HELLO WORLD'
    for s in ['ABC', 'THE QUICK BROWN FOX', '12 34']:
        assert decode(encode(s)) == s.upper(), s
    try:
        encode('hi!')
    except ValueError:
        pass
    else:
        raise AssertionError('expected ValueError on unsupported char')
    print('Part 1 OK')

_t1()

SOS
HELLO WORLD
ABC
THE QUICK BROWN FOX
12 34
Part 1 OK


## Part 2 — On-the-wire representation (storage / transmission)

Real Morse isn't dots and dashes — it's **on/off keying over time**. Standard timing in *units*:

| Element | Units (signal) | Then gap |
|---|---|---|
| dot `.` | 1 ON | 1 OFF (intra-char) |
| dash `-` | 3 ON | 1 OFF (intra-char) |
| between letters | — | 3 OFF |
| between words | — | 7 OFF |

Implement a packed representation suitable for storage/transmission and round-trip it:
- `to_signal(morse) -> list[int]`: a bit/level stream where `1` = key down for one time unit, `0` = key up for one time unit. (e.g. dot = `[1]`, dash = `[1,1,1]`.)
- `from_signal(bits) -> str`: recover the morse string by measuring run lengths.

**Things to discuss:** Why measure run-lengths instead of pattern-matching? What if the clock drifts (a `1` run of length 2 — is it a dot or dash)? How would you store this compactly (RLE? bitpacking into bytes)? Keep `to_signal`/`from_signal` clean — Part 3 will stream these bits across threads.

### Answer to "Things to discuss"

#### 1. Why measure run-lengths instead of pattern-matching?

In an on/off-keyed stream **the information lives in the *durations*, not in fixed bit patterns.** A dot is "key-down for 1 unit", a dash "key-down for 3 units"; the separators are key-*up* for 1 / 3 / 7 units. So the decode has two distinct jobs — **segmentation** (where does each element begin and end) and **classification** (is this run a dot/dash, or an intra-char / letter / word gap).

Run-length encoding (collapse consecutive equal samples into `(level, count)` pairs) is the natural fit:

- **It separates the two jobs.** A level change *is* a segment boundary; the count is the only thing left to classify. Pattern matching (`bits[i+1]==1 and bits[i+2]==1 → dash`) tangles segmentation and classification and hard-codes "1 sample == 1 unit".
- **It's O(n), one pass, allocation-light**, and reduces the stream to just the meaningful quantities (the durations).
- **It generalizes to noise / variable rate.** Classifying a run becomes a numeric comparison against thresholds, which survives jitter (Part 4). A window-matcher keyed to exact sample counts breaks the moment the clock isn't exactly 1 sample/unit, or the operator's speed changes.
- **It composes with streaming.** You can only classify a run once it *ends* (you see the opposite level) — exactly the invariant Part 4's `StreamingDecoder` is built around: buffer the current run, resolve it on the level change.

Pattern-matching is fine for a perfectly clocked, noise-free signal (it's what the hand-rolled `l_from_signal` does), but it's the brittle special case of the run-length view.

#### 2. Clock drift — a `1`-run of length 2: dot or dash?

With an exact decoder (`==1` dot, `==3` dash) a run of 2 is undefined. The robust approach is **threshold classification**: put the boundary at the midpoint between the dot length (1) and dash length (3), i.e. **2 units** — `len < 2 → dot, else → dash`; for gaps the boundaries sit between 1&3 and 3&7, i.e. at **2 and 5**. (That's exactly what `RefStreamingDecoder` uses.)

But a run of *exactly* 2 is genuinely on the fence — a stretched dot or a squeezed dash. You resolve it by making the threshold **adaptive**: track a running estimate of the dot unit (e.g. an EMA of the shortest observed key-down runs) and scale the boundaries to it. A `2` when the dot estimate is ≈1 leans dash; when the estimate has drifted to ≈2 it's a dot. Real telegraphy decoders do exactly this — they continuously adapt to the operator's words-per-minute, because hand-keyed Morse has large timing variance (Farnsworth timing, etc.).

Trade-off to state out loud: fixed thresholds reject noise but mis-decode drift; adaptive thresholds tolerate drift but can mis-segment on an abrupt speed change. Add a guard — if a run is wildly out of range (say > ~1.8× the expected dash), flag it as an error rather than silently guessing.

#### 3. How would you store / transmit it compactly?

The 0/1 signal is the **physical layer**; how small you can go depends on what you control.

- **A Python `list[int]` is far bigger than the data needs.** Each slot is an 8-byte pointer; CPython interns small ints so every `0`/`1` points at one of two shared singletons (so it's ~8 B/sample plus list overhead — a *non*-interned int would be ~28 B each). Either way that's *bytes* per sample to carry *1 bit* of information — fine as an in-process value, never as a wire/disk format.
- **Bit-pack** the stream — it's literally bits, so 8 samples per byte. You must also store the length to drop the final byte's padding:

  ```python
  import numpy as np
  def pack(bits):                                   # bits: list[int] of 0/1
      return len(bits), np.packbits(np.asarray(bits, np.uint8)).tobytes()
  def unpack(n, blob):
      return np.unpackbits(np.frombuffer(blob, np.uint8))[:n].tolist()
  ```
  `SOS` is 27 samples → `ceil(27/8)` = **4 bytes** (+ a small length field), vs ~27 bytes as a raw `bytes`/`bytearray` (one byte per sample) and a few hundred bytes as the Python list.

- **RLE** stores `(level, count)` pairs. Since levels strictly alternate and the signal always starts key-down, you only need the **counts** (level is implied by position). And valid Morse counts are drawn from `{1, 3, 7}`, so you can map them to a tiny codebook and spend ~2 bits each:

  ```python
  def rle(bits):
      runs = []
      for b in bits:
          if runs and runs[-1][0] == b: runs[-1][1] += 1
          else: runs.append([b, 1])
      return [n for _, n in runs]                   # level implied by alternation
  ```
  RLE wins when runs are long; Morse runs are short (≤ 7), so for the raw signal **bit-packing is usually simpler and competitive**. RLE-of-counts pulls ahead once you exploit the `{1,3,7}` alphabet.

- **Best of all: don't ship the time-domain signal.** If you control both ends, transmit the higher-level representation — the morse string, or the original text, or an entropy code over the symbols (Morse is itself a variable-length code optimized for English letter frequencies). You only persist the raw on/off keying when you must (e.g. a recorded RF capture).

**Takeaway for the interview:** name the layers — boxed list (in-memory only) → bit-packed bytes (generic, simple) → RLE / codebook over `{1,3,7}` (Morse-specific) → just send the text (if you own the protocol) — and pick based on what's fixed and how much you control.

> One more, from the prompt's last line — *"keep `to_signal`/`from_signal` clean, Part 3 will stream these bits"*: the RLE decode needs the whole run before it can classify (you must see the `0` that ends a key-down run), so a streaming receiver can't decode sample-by-sample naively — it has to buffer the current run and resolve on each level change. That's the bridge to Part 4.


In [61]:
def to_signal(morse: str) -> list:
    """Morse string (with ' ' and ' / ') -> list of 0/1 time-unit samples."""
    res = []
    stack = []
    for c in morse:
        if c in ['.', '-']:
            stack.append(c)
        elif c == ' ':
            if stack[-1] == '/':
                stack.pop()
                stack.pop()
                stack.append(' / ')
            else:
                stack.append(' ')
        else:
            stack.append(c)
    # print(stack)
    for item in stack:
        if item == ' ':
            res.extend([0]*3)
        elif item == ' / ':
            res.extend([0]*7)
        elif item == '.':
            if res and res[-1] == 1:
                res.append(0)
            res.extend([1])
        elif item == '-':
            if res and res[-1] == 1:
                res.append(0)
            res.extend([1,1,1])
        else:
            raise ValueError(f"unsupported: {item}")
    # print(morse, res)
    return res

def from_signal(bits):
    # run-length encode into (level, length) pairs
    runs = []
    for b in bits:
        if runs and runs[-1][0] == b:
            runs[-1][1] += 1
        else:
            runs.append([b, 1])
    # print(runs)
    out = []
    for level, n in runs:
        if level == 1:
            out.append('.' if n == 1 else '-')
        else:  # gap
            if n == 3:
                out.append(' ')
            elif n == 7:
                out.append(' / ')
            # n == 1 -> intra-char gap, emits nothing
    return ''.join(out)
    
def l_from_signal(bits: list) -> str:
    """Inverse of to_signal: list of 0/1 samples -> morse string."""
    words = []
    stack = []
    i = 0
    while i < len(bits):
        if bits[i] == 1:
            if i + 2 < len(bits):
                if bits[i+1] == 1 and bits[i+2] == 1:
                    words.append('111')
                    i += 3
                    continue
            words.append('1')
            i += 1
        else:
            if i + 6 < len(bits):
                if bits[i+1] == 0 and bits[i+2] == 0 and bits[i+3] == 0 and bits[i+4] == 0 and bits[i+5] == 0 and bits[i+6] == 0:
                    words.append('0000000')
                    i += 7
                    continue
            if i + 2 < len(bits):
                if bits[i+1] == 0 and bits[i+2] == 0:
                    words.append('000')
                    i += 3
                    continue
            i += 1
    dd = {
        '000': ' ',
        '0000000': ' / ',
        '1': '.',
        '111': '-'
    }
    # print(words)
    res = []
    for word in words:
        if word in dd:
            res.append(dd[word])
        else:
            raise ValueError(f"codec doesn't exist: {word}")
    # print(''.join(res))
    return ''.join(res)

In [62]:
# --- Part 2 tests ---
def _t2():
    # 'E' = '.'  -> single key-down unit
    assert to_signal('.') == [1]
    # 'T' = '-'  -> three key-down units
    assert to_signal('-') == [1, 1, 1]
    # 'A' = '.-' -> dot, intra gap(1), dash
    assert to_signal('.-') == [1, 0, 1, 1, 1]
    for s in ['SOS', 'HELLO WORLD', 'THE QUICK BROWN FOX']:
        m = encode(s)
        assert from_signal(to_signal(m)) == m, s
        assert decode(from_signal(to_signal(m))) == s.upper(), s
    print('Part 2 OK')

_t2()

SOS
HELLO WORLD
THE QUICK BROWN FOX
Part 2 OK


## Part 3 — Concurrent transmit / receive

Now simulate a link: a **transmitter** thread emits the signal samples one time-unit at a time onto a shared channel; a **receiver** thread consumes samples and reconstructs the text. This is the classic producer/consumer the interviewer is steering toward.

**Requirements:**
- Use a `queue.Queue` as the channel (bounded — backpressure matters).
- Transmitter pushes samples then signals end-of-stream (sentinel).
- Receiver blocks on the queue, accumulates samples, decodes, and returns the text.
- No busy-waiting, no `sleep`-based synchronization, clean shutdown (join the threads).

**Discussion the interviewer wants:** Why a sentinel vs. `task_done`/`join`? Thread vs. process here (GIL — this is I/O-ish/blocking, so threads are fine; CPU-bound decode of many streams would want processes). How would you handle *multiple* concurrent transmissions multiplexed on one channel?

### Answer to "Things to discuss"

#### 1. Why a sentinel vs. `task_done` / `join`?

They solve *different* problems:

- A **sentinel** ("poison pill") is an **end-of-stream signal that travels in-band**: the producer puts a unique marker after the last real sample, and the consumer's blocking loop `break`s when it sees it. That's exactly what `receive()` needs — it has to know *"no more data is coming"* so it can decode and **return** (which is what lets `rx.join()` complete). Use a unique `object()`, not a magic value like `None`/`-1` that could be valid payload.
- `Queue.join()` / `task_done()` answer a *different* question: *"has all enqueued work been processed?"* `join()` blocks the **producer / main** thread until every `put` has had a matching `task_done`. It does **not** tell the consumer to stop or make `receive()` return — in that pattern consumers are usually infinite-loop daemon threads. And if you forget a `task_done`, `join()` hangs forever.

So here the sentinel is the right tool: a single receiver that must terminate and yield a result. Things to name out loud:
- **Multiple consumers ⇒ one sentinel per consumer** (each consumer pulls exactly one), or re-enqueue the sentinel before breaking so the next consumer sees it.
- **Python 3.13+ has `Queue.shutdown()`** — a built-in alternative to the poison pill that makes pending/future `get`/`put` raise `ShutDown`. Cleaner than a sentinel when you can rely on it.
- This is also how you meet the spec's *no busy-waiting / no sleep-based sync*: the consumer **blocks** on `get()` (the OS wakes it when an item arrives) instead of polling with `time.sleep`, and shutdown is deterministic rather than "sleep long enough and hope."

#### 2. Thread vs. process (the GIL)

For a single link this work is **coordination / blocking-bound, not CPU-bound** — both threads spend almost all their time parked in `queue.get()` / `put()`. The GIL is **released while a thread blocks** on the queue (and on real I/O), so threads give genuine concurrency for *waiting* and barely contend for the GIL. The actual compute (encode / RLE / decode) is microseconds per message, so GIL-serialized Python execution is irrelevant here. Threads also share the `Queue` directly with **no pickling** → right choice.

Switch to **processes** only when decode becomes genuinely CPU-heavy *and* you have many streams (Part 5b): then the GIL would serialize that compute across threads and you'd get no multi-core speedup. `ProcessPoolExecutor` sidesteps it (one GIL per process) at the cost of pickling data across the boundary and needing an importable worker (`morse_workers`). Two nuances worth a sentence: NumPy / C-extension compute often *releases* the GIL (so threads can parallelize it too), and free-threaded CPython (3.13+, experimental) removes the GIL entirely — both shift this calculus.

| | threads | processes |
|---|---|---|
| good for | blocking / I/O, coordination (this link) | CPU-bound Python across cores (batch decode) |
| shared state | yes (the `Queue`) | no — pickle across the boundary |
| GIL | contends only during Python compute | one per process |

#### 3. Multiplexing several transmissions on one channel

This is Part 5a — foreshadow it:

- **Tag every frame with a stream id**: put `(stream_id, bit)` instead of a bare bit. One receiver de-multiplexes into per-stream buffers and decodes each independently:
  ```python
  buffers = defaultdict(list)
  while active_streams:
      sid, item = channel.get()
      if item is SENTINEL:            # one sentinel PER stream
          results[sid] = decode(from_signal(buffers[sid])); active_streams -= 1
      else:
          buffers[sid].append(item)
  ```
- **Fairness / head-of-line blocking**: interleave round-robin (one frame per active stream per pass) so a long message doesn't starve the short ones.
- **Ordering**: a single FIFO preserves *per-stream* order as long as one producer owns each stream; cross-stream interleaving is fine because we demux by id. Add *multiple* consumers and you lose per-stream order unless you shard streams to consumers by id (per-stream affinity / consistent hashing).
- **Backpressure & isolation**: a bounded shared queue means one slow consumer throttles *all* producers, and one stalled stream can block the rest. Fixes: per-stream queues, or buffer raw frames and decode in a separate stage, or a drop/spill policy — decouple *receive* from *decode* so a slow decode of stream A doesn't stall stream B.
- **Framing efficiency**: per-sample `(id, bit)` is simple but heavy on the wire — at scale send length-prefixed frames or whole-message blocks per stream instead.

**Takeaway:** sentinel = in-band "stream ended, return your result" (prefer `Queue.shutdown()` on 3.13+); threads because we're blocking-bound and the GIL is released while waiting; multiplex by tagging frames with a stream id, then handle fairness, ordering, and backpressure isolation.


In [73]:
import queue
import threading
import time

_SENTINEL = object()


def transmit(text: str, channel: 'queue.Queue') -> None:
    """Encode text and push each time-unit sample onto channel, then the sentinel."""
    for c in text:
        channel.put(c)
        # time.sleep(1)
    channel.put(_SENTINEL)


def receive(channel: 'queue.Queue') -> str:
    """Drain channel until sentinel, reconstruct and return the decoded text."""
    res = []
    while (b:=channel.get(timeout=3)):
        # print(b)
        if b == _SENTINEL:
            break
        res.append(b)
    return ''.join(res)

def run_link(text: str) -> str:
    channel = queue.Queue(maxsize=64)
    result = {}
    rx = threading.Thread(target=lambda: result.__setitem__('out', receive(channel)))
    tx = threading.Thread(target=transmit, args=(text, channel))
    rx.start(); tx.start()
    tx.join(); rx.join()
    print(result)
    return result['out']

In [74]:
# --- Part 3 tests ---
def _t3():
    for s in ['SOS', 'HELLO WORLD', 'THE QUICK BROWN FOX']:
        assert run_link(s) == s.upper(), s
    print('Part 3 OK')

_t3()

{'out': 'SOS'}
{'out': 'HELLO WORLD'}
{'out': 'THE QUICK BROWN FOX'}
Part 3 OK


## Part 4 (stretch) — Incremental, noise-tolerant streaming decode

Part 3 buffered *all* samples then decoded. Real receivers can't: the stream may be unbounded and you want letters out as soon as they're unambiguous. The clock also drifts — a dash might arrive as 3 *or* 4 units. Build a stateful decoder.

Implement `StreamingDecoder`:
- `feed(bit) -> str`: consume one 0/1 sample, return any text that just became decodable (often `''`).
- `flush() -> str`: end-of-stream; emit the final pending letter/word.

**The hard parts (this is where they probe):**
- You can't classify a run until it *ends* (you see the opposite level). An ON run only resolves when a `0` arrives; the trailing element only resolves at `flush()`.
- Use *threshold* classification, not `== 1`/`== 3`: split dot/dash at 2 units; split gaps intra/letter/word at 2 and 5. Be ready to argue those midpoints — and how you'd make the threshold *adaptive* (track a running estimate of the dot unit) so it survives an unknown/changing clock rate.
- A letter is only complete when you see a gap `>=` the letter gap. Don't emit on every intra-char gap.
- Edge cases: leading `0`s, two word-gaps in a row, a stream that ends mid-element.

In [ ]:
class StreamingDecoder:
    """Incrementally decode a 0/1 sample stream, tolerant to clock jitter."""

    def __init__(self):
        raise NotImplementedError

    def feed(self, bit: int) -> str:
        """Consume one sample; return any text that just became decodable (may be '')."""
        raise NotImplementedError

    def flush(self) -> str:
        """End of stream: emit the final pending letter/word."""
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
import random


def _jitter(bits, scale, jitter, seed):
    """Stretch each run-length by ~scale with +/- jitter noise: simulates drift."""
    rng = random.Random(seed)
    runs = []
    for b in bits:
        if runs and runs[-1][0] == b:
            runs[-1][1] += 1
        else:
            runs.append([b, 1])
    out = []
    for level, n in runs:
        out += [level] * max(1, round(n * scale + rng.uniform(-jitter, jitter)))
    return out


def _drive(dec, bits):
    out = ''.join(dec.feed(b) for b in bits)
    return out + dec.flush()


def _t4():
    for s in ['SOS', 'HELLO WORLD', 'THE QUICK BROWN FOX']:
        clean = to_signal(encode(s))
        assert _drive(StreamingDecoder(), clean) == s.upper(), ('clean', s)
        noisy = _jitter(clean, scale=1.0, jitter=0.4, seed=7)  # sub-unit timing noise
        assert _drive(StreamingDecoder(), noisy) == s.upper(), ('noisy', s)
    print('Part 4 OK')

_t4()

## Part 5 (stretch) — Multiplexed channels + parallel decode

Now scale out. Two sub-problems:

**(a) Multiplexing one bus.** Several transmitters share a single channel. Each frame is `(stream_id, bit)`; frames from different streams are interleaved (round-robin). One receiver de-multiplexes by `stream_id` into per-stream buffers and decodes each independently.
- `transmit_mux(messages, channel)`: `messages` is `list[(stream_id, text)]`. Interleave frames round-robin; send a per-stream sentinel `(stream_id, _SENTINEL)` when that stream is done.
- `receive_mux(channel, n_streams) -> dict[int, str]`: drain until all `n_streams` have hit their sentinel; return `{stream_id: text}`.
- Discuss: per-stream ordering guarantees, head-of-line blocking, what a bounded queue does to a fast vs slow transmitter (backpressure), how to keep one stalled stream from deadlocking the rest.

**(b) Parallel batch decode.** You have many *recorded* streams (CPU-bound work). Decode them across processes.
- `parallel_decode(streams) -> list[str]`: `streams` is `list[list[int]]`; use `ProcessPoolExecutor`. The worker is `morse_workers.decode_stream` (a real module — see its docstring for why a function defined in the notebook can't be pickled by spawned workers).
- Discuss: GIL (Part 3 was blocking I/O → threads fine; this is CPU-bound → processes), pickling cost of moving streams across the process boundary, chunking, when threads would actually win.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import morse_workers


def transmit_mux(messages, channel: 'queue.Queue') -> None:
    """Interleave (stream_id, bit) frames round-robin; sentinel per stream at end."""
    raise NotImplementedError


def receive_mux(channel: 'queue.Queue', n_streams: int) -> dict:
    """De-multiplex frames by stream_id; return {stream_id: decoded_text}."""
    raise NotImplementedError


def parallel_decode(streams: list) -> list:
    """Decode many recorded 0/1 streams across processes, preserving input order.
    Use morse_workers.decode_stream as the worker (see that module's docstring)."""
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    msgs = [(0, 'SOS'), (1, 'HELLO'), (2, 'FOX')]
    ch = queue.Queue(maxsize=128)
    out = {}
    rx = threading.Thread(target=lambda: out.update(receive_mux(ch, len(msgs))))
    rx.start(); transmit_mux(msgs, ch); rx.join()
    assert out == {0: 'SOS', 1: 'HELLO', 2: 'FOX'}, out
    streams = [to_signal(encode(t)) for _, t in msgs]
    assert parallel_decode(streams) == ['SOS', 'HELLO', 'FOX']
    print('Part 5 OK')

_t5()

## Likely follow-ups (be ready to talk through, maybe code)

- **Noise / clock drift:** classify run-lengths with thresholds instead of exact ==; what threshold and why.
- **Streaming decode:** decode incrementally as samples arrive (don't wait for end-of-stream). Stateful parser.
- **Many channels:** N transmitters, M receivers, a thread pool, ordering guarantees.
- **Compact storage:** bit-pack the 0/1 stream into bytes; RLE; compare sizes.
- **Multiprocessing variant:** decode many recorded streams in parallel — `concurrent.futures.ProcessPoolExecutor`, pickle-ability of the work units.

---
## Reference solutions (don't peek until you've tried)
Run the cell below to define reference implementations under a `ref_` prefix and check them.

In [ ]:
def ref_encode(text):
    words = text.upper().split(' ')
    out = []
    for w in words:
        letters = []
        for ch in w:
            if ch not in MORSE:
                raise ValueError(f'unsupported character: {ch!r}')
            letters.append(MORSE[ch])
        out.append(' '.join(letters))
    return ' / '.join(out)


def ref_decode(morse):
    words = morse.split(' / ')
    return ' '.join(''.join(INV_MORSE[c] for c in w.split(' ') if c) for w in words)


def ref_to_signal(morse):
    bits = []
    words = morse.split(' / ')
    for wi, w in enumerate(words):
        if wi:
            bits += [0] * 7  # word gap
        letters = w.split(' ')
        for li, letter in enumerate(letters):
            if li:
                bits += [0] * 3  # letter gap
            for si, sym in enumerate(letter):
                if si:
                    bits += [0]  # intra-char gap
                bits += [1] * (1 if sym == '.' else 3)
    return bits


def ref_from_signal(bits):
    # run-length encode into (level, length) pairs
    runs = []
    for b in bits:
        if runs and runs[-1][0] == b:
            runs[-1][1] += 1
        else:
            runs.append([b, 1])
    out = []
    for level, n in runs:
        if level == 1:
            out.append('.' if n == 1 else '-')
        else:  # gap
            if n == 3:
                out.append(' ')
            elif n == 7:
                out.append(' / ')
            # n == 1 -> intra-char gap, emits nothing
    return ''.join(out)


def ref_transmit(text, channel):
    for b in ref_to_signal(ref_encode(text)):
        channel.put(b)
    channel.put(_SENTINEL)


def ref_receive(channel):
    bits = []
    while True:
        item = channel.get()
        if item is _SENTINEL:
            break
        bits.append(item)
    return ref_decode(ref_from_signal(bits))


# sanity-check the references
assert ref_decode(ref_encode('Hello World')) == 'HELLO WORLD'
assert ref_decode(ref_from_signal(ref_to_signal(ref_encode('SOS')))) == 'SOS'
_ch = queue.Queue(maxsize=64)
_r = {}
_t = threading.Thread(target=lambda: _r.__setitem__('o', ref_receive(_ch)))
_t.start(); ref_transmit('THE QUICK BROWN FOX', _ch); _t.join()
assert _r['o'] == 'THE QUICK BROWN FOX'
print('reference solutions OK')

In [ ]:
from collections import defaultdict


class RefStreamingDecoder:
    def __init__(self):
        self.level = None
        self.run = 0
        self.letter = []   # symbols of the in-progress letter
        self.text = []     # completed chars (letters + word spaces)

    def _resolve_on(self, n):
        self.letter.append('.' if n < 2 else '-')   # split dot/dash at 2 units

    def _emit_letter(self):
        if self.letter:
            self.text.append(INV_MORSE.get(''.join(self.letter), '?'))
            self.letter = []

    def _resolve_off(self, n):
        if n < 2:
            return                 # intra-char gap -> stay in same letter
        self._emit_letter()        # letter gap OR word gap closes the letter
        if n >= 5:
            self.text.append(' ')  # word gap

    def feed(self, bit):
        if self.level is None:
            self.level, self.run = bit, 1
            return ''
        if bit == self.level:
            self.run += 1
            return ''
        before = len(self.text)
        (self._resolve_on if self.level == 1 else self._resolve_off)(self.run)
        self.level, self.run = bit, 1
        return ''.join(self.text[before:])

    def flush(self):
        before = len(self.text)
        if self.level == 1 and self.run:
            self._resolve_on(self.run)
        self._emit_letter()
        self.level, self.run = None, 0
        return ''.join(self.text[before:])


def ref_transmit_mux(messages, channel):
    pending = [(sid, iter(ref_to_signal(ref_encode(t)))) for sid, t in messages]
    while pending:
        nxt = []
        for sid, it in pending:          # round-robin: one sample per stream per pass
            b = next(it, None)
            if b is None:
                channel.put((sid, _SENTINEL))
            else:
                channel.put((sid, b))
                nxt.append((sid, it))
        pending = nxt


def ref_receive_mux(channel, n_streams):
    buffers, done, active = defaultdict(list), {}, n_streams
    while active:
        sid, item = channel.get()
        if item is _SENTINEL:
            done[sid] = ref_decode(ref_from_signal(buffers[sid]))
            active -= 1
        else:
            buffers[sid].append(item)
    return done


def ref_parallel_decode(streams):
    # worker lives in morse_workers.py so spawned processes can import it
    with ProcessPoolExecutor() as ex:
        return list(ex.map(morse_workers.decode_stream, streams))


# sanity-check the stretch references (run the Part 4 test cell first for _drive)
assert _drive(RefStreamingDecoder(), ref_to_signal(ref_encode('HELLO WORLD'))) == 'HELLO WORLD'
_msgs = [(0, 'SOS'), (1, 'HELLO'), (2, 'FOX')]
_ch2 = queue.Queue(maxsize=128); _o = {}
_rx = threading.Thread(target=lambda: _o.update(ref_receive_mux(_ch2, len(_msgs))))
_rx.start(); ref_transmit_mux(_msgs, _ch2); _rx.join()
assert _o == {0: 'SOS', 1: 'HELLO', 2: 'FOX'}, _o
assert ref_parallel_decode([ref_to_signal(ref_encode(t)) for _, t in _msgs]) == ['SOS', 'HELLO', 'FOX']
print('stretch reference solutions OK')